# 02 - Predictor forward pass and output-head decoding

This notebook confirms how a model's raw output tensor is shaped and then **decoded into Gaussian-mixture parameters**. A randomly initialised tiny U-Net is built on CPU, run on a synthetic patch batch, and its `(B, 3K, H, W)` output is reshaped into the `(B, K, 3, H, W)` amplitude / mean / sigma layout that the predictor's CPU worker consumes.

Modules exercised: `models.get_model`, and the channel-decoding convention used in `pipelines.inference_pipeline.predictor.Predictor._cpu_worker`.

Weights are random, so the values are meaningless; only the shapes and the decoding are under test.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Build a tiny random U-Net

We request a `unet` with `K = 2` Gaussians, hence `out_channels = 3K = 6`, and four input channels. The model is left at its random initialisation and put in eval mode; nothing is loaded from disk.

In [ ]:
from models import get_model

K            = 2
in_channels  = 4
out_channels = 3 * K

model, model_cfg = get_model('unet', in_channels=in_channels, out_channels=out_channels)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print('model config :', type(model_cfg).__name__)
print('in_channels  :', in_channels)
print('out_channels :', out_channels, f'(= 3 x K, K={K})')
print('parameters   :', f'{n_params:,}')


## Forward pass on a synthetic patch batch

A batch of two `32 x 32` patches with four channels is drawn from the seeded RNG. The forward pass runs under `torch.no_grad()` exactly as the predictor does. The output must have shape `(B, 3K, H, W)`.

In [ ]:
B, H, W = 2, 32, 32
images  = torch.randn(B, in_channels, H, W)

with torch.no_grad():
    raw_out = model(images)

print('input  shape:', tuple(images.shape))
print('output shape:', tuple(raw_out.shape))
assert raw_out.shape == (B, out_channels, H, W)
print('output channel count matches 3 x K =', out_channels)


## Decode channels into (amplitude, mean, sigma) per Gaussian

The predictor interprets the flat channel axis as `K` blocks of three: channel `3k+0` is amplitude, `3k+1` is the mean, `3k+2` is sigma. We reshape `(B, 3K, H, W) -> (B, K, 3, H, W)` and verify a round trip back to the flat layout is exact.

In [ ]:
raw_np    = raw_out.numpy()
decoded   = raw_np.reshape(B, K, 3, H, W)
reflatten = decoded.reshape(B, 3 * K, H, W)

print('decoded shape   :', decoded.shape, '(B, K, 3, H, W)')
print('round-trip exact:', np.array_equal(reflatten, raw_np))

amp   = decoded[:, :, 0]
mean  = decoded[:, :, 1]
sigma = decoded[:, :, 2]
print('amplitude block shape:', amp.shape, '(B, K, H, W)')


## Visualise the decoded parameter maps

For the first sample in the batch we display the three parameter maps of each Gaussian slot. Because the weights are random these are noise fields; the point is that the channel split lands the right map in the right slot.

In [ ]:
fig, axes = plt.subplots(K, 3, figsize=(9.0, 3.0 * K))
axes = np.atleast_2d(axes)
names = ['amplitude (a)', 'mean (mu)', 'sigma (s)']

for k in range(K):
    for j in range(3):
        ax = axes[k, j]
        im = ax.imshow(decoded[0, k, j], cmap='viridis')
        ax.set_title(f'g{k+1}  -  {names[j]}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Decoded Gaussian parameter maps (sample 0, random weights)')
fig.tight_layout()
plt.show()


## Expected visual outcome

The output tensor shape must be `(2, 6, 32, 32)`, the reshape to `(2, 2, 3, 32, 32)` must round-trip exactly, and the `K x 3` grid of maps should render without error, one row per Gaussian slot and one column per parameter. The maps are random texture, which is expected from an untrained model.